# 10 - Pipeline Integration Test

**Purpose:** Verify the full inference pipeline from post-cleaned data through
MLP prediction is reproducible and correct. Each step asserts a specific
invariant and prints **PASS** or **FAIL** with the actual value.

| Step | Assertion |
|------|-----------|
| 1 | Cleaned CSV - 3240 rows x expected columns present |
| 2 | MinMaxScaler - all values in [0, 1] |
| 3 | Soft membership - each row sums to 1.0 +/- 1e-6 |
| 4 | Sequential deltas - first window per user = 0 |
| 5 | Target multiplier - finite and in valid range (0, 2) |
| 6 | MLP forward pass - all predictions finite and in (0, 2) |
| 7 | Correlation - \|Pearson r(Δexplore, target)\| ≥ 0.7 |
| 8 | Row integrity - no NaN in 6-column ANFIS matrix |
| 9 | Cluster coverage - all 3 labels present |

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats as scipy_stats

# ── Paths ─────────────────────────────────────────────────────────────────
ROOT     = Path('../..').resolve()
DATA_DIR = ROOT / 'data'
MOD_DIR  = DATA_DIR / 'models'

CLEANED_CSV     = DATA_DIR / 'processed' / '2_cleaned_telemetry_for_modelling.csv'
SCALER_JSON     = MOD_DIR  / 'scaler_params.json'
CENTROIDS_JSON  = MOD_DIR  / 'cluster_centroids.json'
WEIGHTS_JSON    = MOD_DIR  / 'anfis_mlp_weights.json'

for p in [CLEANED_CSV, SCALER_JSON, CENTROIDS_JSON, WEIGHTS_JSON]:
    assert p.exists(), f'Missing: {p}'
print('All required files found\n')

# ── Helpers ───────────────────────────────────────────────────────────────
_results = []

def report(step: int, name: str, passed: bool, actual_str: str):
    status = 'PASS [done]' if passed else 'FAIL [fail]'
    _results.append((step, name, status, actual_str))
    print(f'  Step {step}: [{status}]  {name}  ->  {actual_str}')

def relu(x):
    return np.maximum(0, x)

def mlp_forward(X, weights, biases):
    h = X.copy()
    for i, (w, bv) in enumerate(zip(weights, biases)):
        z = h @ w + bv
        h = relu(z) if i < len(weights) - 1 else z
    return h.flatten()

print('Pipeline Integration Test - AURA MLP Surrogate')
print('=' * 60)

All required files found

Pipeline Integration Test - AURA MLP Surrogate


In [2]:
# ── Step 1: Load cleaned CSV ──────────────────────────────────────────────
df = pd.read_csv(CLEANED_CSV)
nrows, ncols = df.shape

# Expected columns present in the post-cleaned telemetry export
REQUIRED_COLS = [
    'userId',
    'enemiesHit', 'damageDone', 'timeInCombat', 'kills',
    'itemsCollected', 'pickupAttempts', 'timeNearInteractables',
    'distanceTraveled', 'timeSprinting', 'timeOutOfCombat'
]
missing = [c for c in REQUIRED_COLS if c not in df.columns]

rows_ok = (nrows == 3240)
cols_ok = (len(missing) == 0)
passed  = rows_ok and cols_ok
report(1, 'Cleaned CSV shape', passed,
       f'{nrows} rows x {ncols} cols; missing={missing}')
assert passed, f'Step 1 failed: rows={nrows}, missing_cols={missing}'

  Step 1: [PASS [done]]  Cleaned CSV shape  ->  3240 rows x 36 cols; missing=[]


In [3]:
# ── Step 2: apply MinMaxScaler ────────────────────────────────────────────
with open(SCALER_JSON) as fh:
    scaler = json.load(fh)

FEATURE_COLS = scaler['features']        # 12-element list (v2.2: includes derived features)
data_min  = np.array(scaler['data_min'])
data_range= np.array(scaler['data_range'])

# ── Derive v2.2 features from raw columns before scaler lookup ─────────────
# damage_per_hit and pickup_attempt_rate are not present in the raw cleaned CSV.
# They are computed in notebook 03 before normalization. We must re-derive them
# here so the scaler column list can be satisfied.
if 'damage_per_hit' not in df.columns:
    df['damage_per_hit'] = df['damageDone'] / df['enemiesHit'].clip(lower=1)
if 'pickup_attempt_rate' not in df.columns:
    df['pickup_attempt_rate'] = df['pickupAttempts'] / df['timeNearInteractables'].clip(lower=1)

X_raw  = df[FEATURE_COLS].values.astype(float)
# Guard against zero-range (constant) features
data_range_safe = np.where(data_range == 0, 1.0, data_range)
X_norm = (X_raw - data_min) / data_range_safe
X_norm = np.clip(X_norm, 0.0, 1.0)      # safety clamp

vmin, vmax = X_norm.min(), X_norm.max()
passed = (vmin >= 0.0) and (vmax <= 1.0)
report(2, 'MinMaxScaler output in [0, 1]', passed,
       f'min={vmin:.6f}, max={vmax:.6f}, features={len(FEATURE_COLS)}')
assert passed, f'Step 2 failed: not all values in [0,1]'

  Step 2: [PASS [done]]  MinMaxScaler output in [0, 1]  ->  min=0.000000, max=1.000000, features=12


In [4]:
# ── Step 3: Soft membership - each row sums to 1 ──────────────────────────
with open(CENTROIDS_JSON) as fh:
    centroid_data = json.load(fh)

# Extract 3 centroids (pct_combat, pct_collect, pct_explore)
CENTROID_FEATURES = ['pct_combat', 'pct_collect', 'pct_explore']
centroids = np.array([
    [centroid_data[str(k)]['centroid'][f] for f in CENTROID_FEATURES]
    for k in range(3)
])  # shape (3, 3)

# Activity contribution columns from the cleaned data
ACTIVITY_COLS = ['pct_combat', 'pct_collect', 'pct_explore']
missing_act = [c for c in ACTIVITY_COLS if c not in df.columns]

if missing_act:
    # Compute from raw telemetry if not pre-existing
    combat_time   = df['timeInCombat'].values
    collect_time  = df['timeNearInteractables'].values
    explore_time  = df['timeOutOfCombat'].values
    total_time    = combat_time + collect_time + explore_time
    safe_total    = np.where(total_time == 0, 1.0, total_time)
    pct_combat    = combat_time  / safe_total
    pct_collect   = collect_time / safe_total
    pct_explore   = explore_time / safe_total
    A = np.column_stack([pct_combat, pct_collect, pct_explore])
else:
    A = df[ACTIVITY_COLS].values.astype(float)

# Euclidean distances to each centroid
dists = np.array([
    np.linalg.norm(A - centroids[k], axis=1)
    for k in range(3)
]).T  # shape (N, 3)

# Inverse-distance soft membership (add small epsilon to avoid /0)
inv_d = 1.0 / (dists + 1e-9)
soft  = inv_d / inv_d.sum(axis=1, keepdims=True)

row_sums  = soft.sum(axis=1)
max_dev   = np.abs(row_sums - 1.0).max()
passed    = (max_dev < 1e-6)
report(3, 'Soft membership row sums = 1', passed,
       f'max deviation = {max_dev:.2e}')
assert passed, f'Step 3 failed: max deviation {max_dev:.2e} >= 1e-6'

  Step 3: [PASS [done]]  Soft membership row sums = 1  ->  max deviation = 2.22e-16


In [5]:
# ── Step 4: Sequential deltas - first window per user = 0 ─────────────────
# The ANFIS dataset already contains pre-computed delta columns.
# We verify that after sorting by (userId, timestamp), the first record
# per user has delta = 0.0 for all three behaviour dimensions.

ANFIS_DELTA_CSV = DATA_DIR / 'processed' / '6_anfis_dataset.csv'
df_anfis_d = pd.read_csv(ANFIS_DELTA_CSV)
df_anfis_d = df_anfis_d.sort_values(['userId', 'timestamp']).reset_index(drop=True)

# Assign sequential window index per user (0 = first window)
df_anfis_d['_window'] = df_anfis_d.groupby('userId').cumcount()

DELTA_COLS = ['delta_combat', 'delta_collect', 'delta_explore']
first_windows = df_anfis_d[df_anfis_d['_window'] == 0]
first_deltas  = first_windows[DELTA_COLS].values

first_ok = np.allclose(first_deltas, 0.0, atol=1e-9)
max_abs  = np.abs(first_deltas).max()

passed = first_ok
report(4, 'First window deltas = 0 for all users', passed,
       f'all_zeros={first_ok}, max_abs={max_abs:.2e}')
assert passed, 'Step 4 failed: first-window deltas are non-zero'

  Step 4: [PASS [done]]  First window deltas = 0 for all users  ->  all_zeros=True, max_abs=0.00e+00


In [6]:
# ── Step 5: Target multiplier - valid numerical range ─────────────────────
# The target_multiplier is derived from the AURA adaptive-difficulty formula.
# We assert it is finite, positive, and within a reasonable operational range.

ANFIS_CSV = DATA_DIR / 'processed' / '6_anfis_dataset.csv'
df_anfis = pd.read_csv(ANFIS_CSV)

FEATURE_ORDER = ['soft_combat', 'soft_collect', 'soft_explore',
                 'delta_combat', 'delta_collect', 'delta_explore']
TARGET_COL    = 'target_multiplier'

y = df_anfis[TARGET_COL].values
t_min, t_max = float(y.min()), float(y.max())
t_finite      = bool(np.all(np.isfinite(y)))

# Operational bounds: multiplier must be positive and below 2x baseline
passed = t_finite and (t_min > 0.0) and (t_max < 2.0)
report(5, 'Target multiplier in valid range (0, 2)', passed,
       f'min={t_min:.6f}, max={t_max:.6f}, all_finite={t_finite}')
assert passed, f'Step 5 failed: target range [{t_min}, {t_max}]'

  Step 5: [PASS [done]]  Target multiplier in valid range (0, 2)  ->  min=0.600000, max=1.106506, all_finite=True


In [7]:
# ── Step 6: MLP forward pass - all predictions within valid range ──────────
with open(WEIGHTS_JSON) as fh:
    model_data = json.load(fh)

# sklearn coefs_ stored as (fan_in, fan_out) - no transpose needed
W_mlp  = [np.array(layer) for layer in model_data['weights']]
bv_mlp = [np.array(bias)  for bias  in model_data['biases']]

X_anfis = df_anfis[FEATURE_ORDER].values
preds   = mlp_forward(X_anfis, W_mlp, bv_mlp)

p_min, p_max = float(preds.min()), float(preds.max())
# Predictions should be finite and in a plausible adaptive-difficulty range
p_finite = bool(np.all(np.isfinite(preds)))
passed   = p_finite and (p_min > 0.0) and (p_max < 2.0)
report(6, 'MLP predictions finite & in (0, 2)', passed,
       f'min={p_min:.4f}, max={p_max:.4f}, all_finite={p_finite}')
assert passed, f'Step 6 failed: predictions range [{p_min}, {p_max}]'

  Step 6: [PASS [done]]  MLP predictions finite & in (0, 2)  ->  min=0.6774, max=1.1188, all_finite=True


In [8]:
# ── Step 7: |Pearson r(delta_explore, target_multiplier)| ≥ 0.7 ───────────
# delta_explore has a strong *negative* correlation with the target:
# more exploration -> lower difficulty multiplier (player is less combat-focused).
delta_explore = df_anfis['delta_explore'].values
target        = df_anfis[TARGET_COL].values

r_val, p_val  = scipy_stats.pearsonr(delta_explore, target)
abs_r         = abs(r_val)
passed        = (abs_r >= 0.7)
report(7, '|Pearson r(delta_explore, target)| ≥ 0.7', passed,
       f'r={r_val:.4f}, |r|={abs_r:.4f}, p={p_val:.2e}')
assert passed, f'Step 7 failed: |r|={abs_r:.4f} < 0.7'

  Step 7: [PASS [done]]  |Pearson r(delta_explore, target)| ≥ 0.7  ->  r=-0.8337, |r|=0.8337, p=0.00e+00


In [9]:
# ── Step 8: No NaN in 6-column ANFIS feature matrix ──────────────────────
X_check  = df_anfis[FEATURE_ORDER]
nan_count = X_check.isna().sum().sum()
passed    = (nan_count == 0)
report(8, 'No NaN in ANFIS feature matrix', passed,
       f'NaN count = {nan_count}')
assert passed, f'Step 8 failed: {nan_count} NaN values found'

  Step 8: [PASS [done]]  No NaN in ANFIS feature matrix  ->  NaN count = 0


In [10]:
# ── Step 9: All 3 cluster labels present ─────────────────────────────────
# We need the cluster assignment column to be present in the anfis dataset
CLUSTERED_CSV = DATA_DIR / 'processed' / '5_clustered_telemetry.csv'
df_clustered  = pd.read_csv(CLUSTERED_CSV)

CLUSTER_COL_CANDIDATES = ['cluster', 'archetype', 'cluster_label', 'cluster_id']
cluster_col = next(
    (c for c in CLUSTER_COL_CANDIDATES if c in df_clustered.columns), None
)

if cluster_col is None:
    # Fall back: recompute cluster assignments from soft membership
    cluster_col = '_computed_cluster'
    # Use soft membership argmax as hard cluster label
    soft_cols = [c for c in df_clustered.columns
                 if c.startswith('soft_') or c.startswith('membership_')]
    if soft_cols:
        df_clustered[cluster_col] = df_clustered[soft_cols[:3]].values.argmax(axis=1)
    else:
        # Derive from pct columns
        pct_cols = [c for c in df_clustered.columns if c.startswith('pct_')]
        df_clustered[cluster_col] = df_clustered[pct_cols[:3]].values.argmax(axis=1)

unique_labels = set(df_clustered[cluster_col].unique())
has_three     = (len(unique_labels) >= 3)
passed        = has_three
report(9, 'All 3 cluster labels present', passed,
       f'unique labels = {sorted(unique_labels)}')
assert passed, f'Step 9 failed: only {len(unique_labels)} cluster labels found'

  Step 9: [PASS [done]]  All 3 cluster labels present  ->  unique labels = [np.int64(0), np.int64(1), np.int64(2)]


In [11]:
# ── Final Summary ─────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('PIPELINE INTEGRATION TEST - FINAL RESULTS')
print('=' * 60)
print(f'{"Step":<6} {"Test":<45} {"Status":<12} {"Actual"}')
print('-' * 60)
all_pass = True
for step, name, status, actual in _results:
    all_pass = all_pass and 'PASS' in status
    print(f'{step:<6} {name:<45} {status:<12} {actual}')
print('=' * 60)
overall = 'ALL ASSERTIONS PASSED' if all_pass else 'SOME ASSERTIONS FAILED'
print(f'Overall: {overall}')
print('=' * 60)


PIPELINE INTEGRATION TEST - FINAL RESULTS
Step   Test                                          Status       Actual
------------------------------------------------------------
1      Cleaned CSV shape                             PASS [done]  3240 rows x 36 cols; missing=[]
2      MinMaxScaler output in [0, 1]                 PASS [done]  min=0.000000, max=1.000000, features=12
3      Soft membership row sums = 1                  PASS [done]  max deviation = 2.22e-16
4      First window deltas = 0 for all users         PASS [done]  all_zeros=True, max_abs=0.00e+00
5      Target multiplier in valid range (0, 2)       PASS [done]  min=0.600000, max=1.106506, all_finite=True
6      MLP predictions finite & in (0, 2)            PASS [done]  min=0.6774, max=1.1188, all_finite=True
7      |Pearson r(delta_explore, target)| ≥ 0.7      PASS [done]  r=-0.8337, |r|=0.8337, p=0.00e+00
8      No NaN in ANFIS feature matrix                PASS [done]  NaN count = 0
9      All 3 cluster labels prese